In [2]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import glob
import sys
sys.path.append('/modules/rhel8/user-apps/fou-modules/fou-hi/toolbox/dev')
import toolbox

/home/nilsmk/.conda/envs/nils_production-10-2022_rhel8/lib/python3.9/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
def latlon2xy(lat,lon,lat_arr,lon_arr,mask=None,maxoff=None,roundvals=True):
    import numpy as np
    from toolbox.utils.BFS import BFS
    a = abs( lat_arr-lat ) + abs( lon_arr-lon )
    ypos, xpos = np.unravel_index(a.argmin(), a.shape)
    if roundvals:
        xpos = int(np.round(xpos))
        ypos = int(np.round(ypos))
    if mask is not None:
        # NOTE: Mask should be 0 over land!!
        ypos_off, xpos_off = BFS(ypos, xpos, mask)
        if maxoff:
            if (np.sqrt((np.abs(xpos-xpos_off)**2)+(np.abs(ypos-ypos_off)**2)) > maxoff):
                print('offset too big')
                return None
            else:
                return xpos_off, ypos_off
        else:
            return xpos_off, ypos_off
    return xpos, ypos


In [ ]:
# Define stuff:
# Input observations:
idir_obs          = '/lustre/storeB/users/nilsmk/stormrisk'
# Output data file:
ofile_nc          = os.path.join(idir_obs, 'anemoi_manuscript_obs_and_models.nc')
# Read observation dataset:
ifile_tseries_obs = os.path.join(idir_obs, 'aggregated_water_level_observations_with_pytide_prediction_dataset_extended_nor_stations_with_4_nora-surge_exps.nc')
ds_obs            = xr.open_dataset(ifile_tseries_obs)
# Limit dataset to relevant time period:
ds_obs            = ds_obs.sel(time=slice('1980-01-01', '2023-01-01'))
# Select only hourly data:
ds_obs            = ds_obs.sel(time=ds_obs['time'].dt.minute == 0)
# Only select the following variables:
ds_obs            = ds_obs[['stationid', 'latitude', 'longitude', 'observation', 'pytide_prediction', 'q_flag', 'nora3c']]
if not os.path.exists(ofile_nc):
    # Convert to dataframe:
    df_obs            = ds_obs.to_dataframe()

/home/nilsmk/.conda/envs/nils_production-10-2022_rhel8/lib/python3.9/site-packages/xarray/backends/plugins.py:64: RuntimeWarning: Engine 'gini' loading failed:
cannot import name 'deprecated' from 'typing_extensions' (/home/nilsmk/.conda/envs/nils_production-10-2022_rhel8/lib/python3.9/site-packages/typing_extensions.py)
  warnings.warn(f"Engine {name!r} loading failed:\n{ex}", RuntimeWarning)


In [ ]:
if not os.path.exists(ofile_nc):
    # Create new column for observation_surge:
    df_obs['observation_surge'] = df_obs['observation'] - df_obs['pytide_prediction']
    # Rename nora3c to nora_surge:
    df_obs = df_obs.rename(columns={'nora3c': 'nora_surge'})

- First, read the nora3 corrected and put into new column
- Read in the anemoi runs, give good names
- Create a new column in the df with the anemoi run name
- Loop over the stations (or time?) and select the station data and put it into the correct index of the df. __Convert m to cm__
- Redo for more anemoi runs
- Store df to file
- Do statistics and plotting

In [ ]:
if not os.path.exists(ofile_nc):
    #
    nora_surge_corr = '/lustre/storeB/users/nilsmk/stormrisk/nora3_v2_corrected/{}/ocean_his.nc'
    files_corr = sorted(glob.glob(nora_surge_corr.format('*')))
    #
    # Create a new column in the dataframe for the corrected nora surge:
    df_obs['nora_surge_corrected'] = np.nan
    #
    # Loop over files and read data, and put into the correct station and time in the dataframe. 
    # Ensure that the units are in cm (not m as in the netcdf files).
    # Calculate the x- and y-indices for each station once based on the first file. Ensure that the selected index contains valid data, else search nearby indices for valid data.
    ds_corr = xr.open_dataset(files_corr[0])
    maskr = np.where(np.isnan(ds_corr.zeta[0,:].values), 0, 1)
    # Precompute station grid indices based on the first file and the stations in df_obs
    station_indices = {}
    for sid,station_id in enumerate(ds_obs.stationid.values):
        print(f'Computing indices for station ID: {station_id}, no. {sid+1} of {len(ds_obs.stationid.values)}')
        lat_station = ds_obs.latitude.sel(station=np.where(ds_obs.stationid==station_id)[0][0]).values
        lon_station = ds_obs.longitude.sel(station=np.where(ds_obs.stationid==station_id)[0][0]).values
        x_index, y_index = latlon2xy(lat_station, lon_station, ds_corr['lat_rho'].values, ds_corr['lon_rho'].values, mask=maskr, roundvals=True)
        station_indices[station_id] = (sid, y_index, x_index)
        print(f'-->Station ID {station_id} assigned to grid indices (y: {y_index}, x: {x_index}). Original lat: {lat_station}, lon: {lon_station}, Grid lat: {ds_corr["lat_rho"][y_index, x_index].values}, lon: {ds_corr["lon_rho"][y_index, x_index].values}')
    ds_corr.close()

In [7]:
if os.path.exists(ofile_nc):
    print(f'Output file {ofile_nc} already exists. Will read data into dataframe from this file.')
    ds_out = xr.open_dataset(ofile_nc)
    df_obs = ds_out.to_dataframe()
    ds_out.close()
else:
    for i, file in enumerate(files_corr):
        print(f'Reading file {i+1} of {len(files_corr)}: {file}')
        ds_corr = xr.open_dataset(file)
        zeta_data_all = ds_corr['zeta'].values * 100.0  # Convert from m to cm
        time_data = ds_corr['ocean_time'].values

        # Loop over stations and extract data
        for station_id, (sid, y_index, x_index) in list(station_indices.items()):
            print(f'Processing station {sid+1} of {len(station_indices)}')
            
            # Extract data for specific station
            zeta_data = zeta_data_all[:, y_index, x_index]
            
            # Create a temporary dataframe
            df_temp = pd.DataFrame({
                'time': pd.to_datetime(time_data), 
                'nora_surge_corrected': zeta_data
            })
            df_temp.set_index('time', inplace=True)
            
            # --- THE FIX ---
            # 1. Identify rows in df_obs that match the current Station
            # 2. AND match the Time points present in the current file
            mask = (df_obs.index.get_level_values('station') == sid) & \
                (df_obs.index.get_level_values('time').isin(df_temp.index))
            
            # Only update the rows where the mask is True
            # We use .values on the right side to ignore index alignment issues during assignment
            df_obs.loc[mask, 'nora_surge_corrected'] = df_temp['nora_surge_corrected'].values
            
        del zeta_data_all
        ds_corr.close()
    # Save df_obs to a new netcdf file
    df_obs.to_xarray().to_netcdf(ofile_nc)

Output file /lustre/storeB/users/nilsmk/stormrisk/anemoi_manuscript_obs_and_models.nc already exists. Will read data into dataframe from this file.


In [8]:
# Create a time slice for 1990 from df_obs for plotting
# df_obs_1990 = df_obs.loc[(df_obs.index.get_level_values('time') >= '1990-01-01') & (df_obs.index.get_level_values('time') < '1991-06-01')]


In [9]:
# for i in range(106):
#     df_obs_1990.nora_surge_corrected[i].plot()

In [10]:
# Now add the anemoi experiments:
idir_anemoi = '/lustre/storeB/users/nilsmk/anemoi-datasets/output/{}/inference'
# Exps: (foldername, shortname, nc_filepattern)
exps_anemoi = [('longrun_10years_newgraph_90s_uncorr', 'train16yr_uncorr_10d_100k', '*_240h_step100000.nc'), 
               ('longrun_10years_newgraph_90s_corrected', 'train16yr_corr_10d_100k', '*_240h_step100000.nc'),
            #    ('longrun_graph10.3_1980-2012', 'train30yr_10d_183k', '*_10day_step183000.nc'), 
            #    ('longrun_graph10.3_1980-2012', 'train30yr_1y_149k', '*_1year_step149000.nc'), 
               ('longrun_5years_newgraph_90s_uncorr', 'train5yr_uncorr_10d_100k', '*_240h_step100000.nc')]
            # Do inference for 10 days for 5yr train based on 100k, and also 25k
            # Do inference for 1 year for all runs
            # Do continous inference from 2010-2018 for all?


# Evaluate the 10days (240h) inference for each exp (only 2013). 
# Also add the 1year inference from longrun_1980-2012 (for years 2013 - 2022)

In [ ]:
# Loop over all anemoi exps, read data, and add to dataframe.
for exp in exps_anemoi:
    foldername, shortname, filepattern = exp
    print(f'Processing Anemoi experiment: {foldername}, shortname: {shortname}')
    idir_exp = idir_anemoi.format(foldername)
    files_exp = sorted(glob.glob(os.path.join(idir_exp, filepattern)))
    for i, file in enumerate(files_exp):
        print(f'Reading file {i+1} of {len(files_exp)}: {file}')
        ds_anemoi = xr.open_dataset(file)
        _zeta = ds_anemoi.zeta
        _lat = ds_anemoi.latitude
        _lon = ds_anemoi.longitude
        time = ds_anemoi.time
        #
        zeta = np.reshape(_zeta.values, (_zeta.shape[0], 570, 1014))
        lat = np.reshape(_lat.values, (570, 1014))
        lon = np.reshape(_lon.values, (570, 1014))
        # If i is 0, find positions for all stations, define new column in dataframe
        if i == 0:
            print('Computing station indices for Anemoi data...')
            df_obs[shortname] = np.nan
            station_indices_anemoi = {}
            maskr_anemoi = np.where(np.isnan(zeta[0,:]), 0, 1)
            # Loop over all stations:
            for sid,station_id in enumerate(ds_obs.stationid.values):
                # print(f'Computing indices for station ID: {station_id}, no. {sid+1} of {len(ds_obs.stationid.values)}')
                lat_station = ds_obs.latitude.sel(station=np.where(ds_obs.stationid==station_id)[0][0]).values
                lon_station = ds_obs.longitude.sel(station=np.where(ds_obs.stationid==station_id)[0][0]).values
                x_index, y_index = latlon2xy(lat_station, lon_station, lat, lon, mask=maskr_anemoi, roundvals=True)
                station_indices_anemoi[station_id] = (sid, y_index, x_index)
                # print(f'-->Station ID {station_id} assigned to grid indices (y: {y_index}, x: {x_index}). Original lat: {lat_station}, lon: {lon_station}, Grid lat: {lat[y_index, x_index]}, lon: {lon[y_index, x_index]}')
        # Loop over all stations and extract data and put into df_obs:
        for station_id, (sid, y_index, x_index) in list(station_indices_anemoi.items()):
            # print(f'Processing station {sid+1} of {len(station_indices_anemoi)}')
            # Extract data for specific station
            zeta_data = zeta[:, y_index, x_index]
            # Create a temporary dataframe
            df_temp = pd.DataFrame({
                'time': pd.to_datetime(time.values), 
                shortname: zeta_data
            })
            df_temp.set_index('time', inplace=True)
            # --- THE FIX ---
            # 1. Identify rows in df_obs that match the current Station
            # 2. AND match the Time points present in the current file
            mask = (df_obs.index.get_level_values('station') == sid) & \
                (df_obs.index.get_level_values('time').isin(df_temp.index))
            # Only update the rows where the mask is True
            # We use .values on the right side to ignore index alignment issues during assignment
            df_obs.loc[mask, shortname] = df_temp[shortname].values
        ds_anemoi.close()


Processing Anemoi experiment: longrun_10years_newgraph_90s_uncorr, shortname: train16yr_uncorr_10d_100k
Reading file 1 of 37: /lustre/storeB/users/nilsmk/anemoi-datasets/output/longrun_10years_newgraph_90s_uncorr/inference/2013-01-01_240h_step100000.nc
Computing station indices for Anemoi data...
Processing station 1 of 106
Processing station 2 of 106
Processing station 3 of 106
Processing station 4 of 106
Processing station 5 of 106
Processing station 6 of 106
Processing station 7 of 106
Processing station 8 of 106
Processing station 9 of 106
Processing station 10 of 106
Processing station 11 of 106
Processing station 12 of 106
Processing station 13 of 106
Processing station 14 of 106
Processing station 15 of 106
Processing station 16 of 106
Processing station 17 of 106
Processing station 18 of 106
Processing station 19 of 106
Processing station 20 of 106
Processing station 21 of 106
Processing station 22 of 106
Processing station 23 of 106
Processing station 24 of 106
Processing stat